# COCO Project 2026 - Assignment

This file is the detailed and interactive assignment file. It explains the functions and contains some sample code. Use it to familiarize yourself. It is not regarded during submission.

## Introduction

Vertical Takeoff, Vertical Landing (VTVL) is a form of takeoff and landing for rockets which can allow for reusability. Perhaps the most widely known and successful reusable VTVL vehicle is [SpaceX's Falcon 9](https://www.spacex.com/vehicles/falcon-9/) first stage. Reusable rocket technology has been studied extensively in the last two decades, thanks to its many advantages over non-reusable flight systems. These include drastically lowered launch costs, rapid reusability, and reducing the environmental impacts associated with manufacturing.

In this project, you will simulate the final phase of a VTVL misson profile, which is the vertical landing phase. A [Gymnasium](https://gymnasium.farama.org) environment has been developed which simulates the simplified 2D dynamics of a VTVL rocket using the [Box2D](https://box2d.org) physics engine.

The goal of this project is to work in this environment and to develop controllers which are able to guide the descent of the rocket and bring it safely to the landing spot.

## Rocket Dynamics, State, and Action Spaces


A free body diagram illustrating the forces acting on the simplified 2D rocket is shown below. Note that we do not consider air resistance or aerodynamic effects in the 2D model, for simplicity.


<img src="https://gitlab.ethz.ch/bsaverio/coco-project/-/raw/main/images/Rocket_Dynamics.png" width=400/>

### Actions
Three inputs are available for steering the rocket:
- $F_E$: the thrust produced by the main engine in Newtons (N), acting directly on the rocket body at the point where the nozzle pivots;
- $F_S$: the thrust produced by the side gas thrusters in Newtons (N), defined as the difference $F_L - F_R$, acting at a distance $l_2$ from the rocket center of mass;
- $\phi$: the angle of the nozzle with respect to the rocket body in radians (rad), which changes the direction of $F_E$. It can be discontinuous and it has instant response up to the 60 fps frame rate of our model (sampling time of the environment).

The three inputs are provided to the environment as an action:

$$\textbf{u} = \begin{bmatrix}F_E & F_S & \phi\end{bmatrix}$$

which is applied to the environment at the 60 fps frame rate. In other words, your controller must provide 60 actions per second of simulation time. These could be new actions, or the same action could be applied across multiple simulation steps.

Unlike the real world, there is no hard constraint on how long your controller has to produce a next action, although we should be able to run your code in a reasonable amount of time. In our own testing, a well performing optimization-based method took much less than 1 minute to run.

<font color='red'>IMPORTANT</font>: The controller should return *normalized* actions, as a fraction of full scale. The main engine thrust is clipped to $[0, 1]$, while the side engine thrust and nozzle angle is clipped to $[-1, 1]$. The environment then scales these to the appropriate ranges (see *Limits on the Actuators*).


### State Variables
The state of the rocket is defined by the following variables:
- $(x, y)$: the 2D position of the rocket center of mass in meters (m);
- $(\dot{x}, \dot{y})$: the velocity of the rocket in meters per second (m/s);
- $\theta$: the angle of the rocket in radians (rad);
- $\dot{\theta}$: the angular velocity of the rocket in radians per second (rad/s);
- $c_L, c_R$: binary variables indicating whether the left or right legs are in contact with the environment, respectively (equal to 0 otherwise).

The environment returns these variables as an observation:
$$\textbf{x} = \begin{bmatrix}x & y & \dot{x} & \dot{y} & \theta & \dot{\theta}& c_L & c_R\end{bmatrix}$$

The last two variables ($c_L, c_R$) may generally be ignored in your control implementation, but may be useful e.g. for turning off one or more actions once ground contact is made.

<font color='red'>IMPORTANT</font>: All variables are with respect to world frame, which is located in the *bottom-left* corner of the simulation window. The x-axis increases going right (horizontally) along the window and the y-axis increases going up (vertically). Positive angles are measured counter-clockwise from the y-axis.

## PID Controller

The following code initializes a basic PID-type controller that can be used to steer the rocket from gentle initial conditions.

In [ ]:
from coco_rocket_lander.algs import PID_Benchmark

engine_pid_params = [10, 0, 10]
engine_vector_pid_params = [0.085, 0.001, 10.55]
side_engine_pid_params = [5, 0, 6]

pid = PID_Benchmark(engine_pid_params, engine_vector_pid_params, side_engine_pid_params)

## Simulator Usage

We now use the PID controller to compute the next actions in the simulation. The following code also illustrates how to interact with the simulator environment.

In [ ]:
import gymnasium as gym
import numpy as np

import coco_rocket_lander  # need to import to call gym.make()

# make environment and wrap video so that we can replay them later
env = gym.make("coco_rocket_lander/RocketLander-v0", render_mode="rgb_array", args={})
env = gym.wrappers.RecordVideo(env, 'video', episode_trigger = lambda x: True, name_prefix="pid_example")

obs, info = env.reset(seed=0)  # specify a random seed for consistency

# simulate
while True:
    landing_position = env.unwrapped.get_landing_position()  # (x, y, theta) in [m, m, radians]

    # offset so that (0, 0) is at landing position
    pid_obs = obs
    pid_obs[0] = obs[0] - landing_position[0]
    pid_obs[1] = obs[1] - landing_position[1]

    # get action
    action = np.array(pid.pid_algorithm(pid_obs))

    next_obs, rewards, done, _, info = env.step(action)

    # check if simulation ended
    if done:
        break

    # update observation
    obs = next_obs

env.close()  # video saved at this step

### View the Video

The above code specifies to save the video in a folder called `video`. We can open this folder to look at the recorded episode as follows. First, we define a `show_video` method for convenience:


In [ ]:
from coco_rocket_lander.utils import show_video

Then we invoke this method to play the first video in the `video` folder:

In [ ]:
show_video('pid_example')

## Linearized Model of the Rocket Problem

A basic linearized model of the rocket dynamics has been derived and is provided to you, for the discrete-time update equation of the form:

$$\textbf{x}_{k+1} = A \textbf{x}_{k} + B\textbf{u}_{k}.$$
<!-- $$\dot{\textbf{x}} = A \textbf{x} + B\textbf{u}$$ -->
 <!-- $$\textbf{x}_{t+1} = A \textbf{x}_{t} + B\textbf{u}_{t}.$$ -->

Note:
- This equation assumes that we have discarded the last two elements ($c_L, c_R$) of the state observation $\textbf{x}$ returned by the environment;
- The $A$ matrix has shape $\mathbb{R}^{6\times 6}$;
- The $B$ matrix has shape $\mathbb{R}^{6\times3}$;
- We linearize around the upright equilibrium with action $\tilde{\textbf{u}} = \begin{bmatrix} mg & 0 & 0 \end{bmatrix}$

Moreover, we scale $B$ such that *normalized actions* (as a fraction of full scale) satisfy the update equation. The normalization of $B$ is performed as follows:

  $$B = B_\textrm{unnormalized}
  \begin{bmatrix}
    F_{E, \textrm{max}} & 0 & 0\\
    0 & F_{S, \textrm{max}} & 0\\
    0 & 0 & \phi_{\textrm{max}}
  \end{bmatrix}$$

where $B_\textrm{unnormalized}$ is simply the usual linearized, discrete-time input matrix obtained from the system dynamics. The maximum values used for scaling are given in the following section, *Limits on the Actuators*.

The following code demonstrates how to obtain the system matrices:

In [ ]:
import gymnasium as gym

from coco_rocket_lander.env import SystemModel

env = gym.make("coco_rocket_lander/RocketLander-v0", render_mode="rgb_array", args={})

model = SystemModel(env.unwrapped)

# by default, linearize around upright equilibrium with F_E = m*g
model.calculate_linear_system_matrices()

# you are free to change the sampling time as you wish
model.discretize_system_matrices(sample_time=0.1)
A, B = model.get_discrete_linear_system_matrices()

print("A:\n", A.round(3), "\n"*2, "B:\n", B.round(3))


## Specifications of the Control Problem

### Limits on the Actuators

The following limits should be assumed for the normalized actions input to the enivornment:

- $F_E \in [0, 1]$;
- $F_S \in [-1, 1]$;
- $\phi \in [-1 ,1]$

Actions outside this range will be clipped. These are then multiplied by the actual maximum thrust/angle limits to get the value which is applied to the environment. The maximum values can be queried as follows:

In [ ]:
print(f"Main engine max. thrust (F_E,max): {env.unwrapped.cfg.main_engine_thrust}")
print(f"Side engine max. thrust (F_S,max): {env.unwrapped.cfg.side_engine_thrust}")
print(f"Max. absolute nozzle angle (phi_max) (in radians): {env.unwrapped.cfg.max_nozzle_angle}")

The limits can also be directly queried from the environment:

In [ ]:
print(f"Main engine normalized thrust limits: {env.unwrapped.cfg.main_engine_thrust_limits}")
print(f"Side engine normalized thrust limits: {env.unwrapped.cfg.side_engine_thrust_limits}")
print(f"Nozzle normalized angle limits: {env.unwrapped.cfg.nozzle_angle_limits}")

### Limits on the State
The following limits should be assumed for the state:

- $x \in [0, 33.333]$;
- $y \in [0, 26.666]$;
- $\theta \in [-0.6108, 0.6108]$ (or $±35°$)

These are the *maximal* limits for the simulation, and the simulation will terminate if these are exceeded. Your controller should at least consider these limits on the state space.

There are no enforced limits on the linear or angular velocity. Of course, the simulator steps forwards at 60 frames per second and there is some velocity at which the rocket will exit the allowable state space in a single timestep.

<font color='red'>IMPORTANT</font>: Note that the environment exclusively uses *radians* for angles.

These values can also be directly queried from the environment:

In [ ]:
print(f"State x limits: {[0, env.unwrapped.cfg.width]}")
print(f"State y limits: {[0, env.unwrapped.cfg.height]}")
print(f"State angle limits (in radians): {[-env.unwrapped.cfg.theta_limit, env.unwrapped.cfg.theta_limit]}")

### Landing Constraints
By default, the landing barge is placed halfway across the screen and 1/8th of the way up, unless changed on initialization (see *Arguments Provided to the Environment* below). The landing position can be queried from the environment at runtime as:

In [ ]:
landing_position = env.unwrapped.get_landing_position()  # 3-tuple (x_des, y_des, theta_des)
print(f"Landing position (x, y, theta): {landing_position}")

Where $x_\mathrm{des}$, $y_\mathrm{des}$ and $\theta_\mathrm{des}$ is the desired 2D position and angle of the rocket such that it lands on the center of the barge.

<font color='red'>IMPORTANT</font>: For simplicity of implementation, we return the position of the rocket center of mass *above* the barge that the rocket center of mass should be at for a successful landing. We also subtract a small fudge factor of 0.15m such that the rocket legs actually trigger a contact with the barge.

#### Successful Landings

If the rocket legs make contact with the sea, or if the rocket body makes contact with anything, we immediately terminate the environment. The rocket legs may otherwise make contact with any part of the landing platform.

If both rocket legs make contact with the landing platform and the rocket stops moving, the landing will be considered successful. The environment will return `done = True` and a reward of +100. If the environment terminates for any other reason, the environment will still return `done = True` but the reward will be -100.

### Arguments Provided to the Environment

Initialization arguments can be provided to the environment in a dictionary. We've defined several arguments which a user may provide to test different aspects of the simulation. The following is an excerpt of the relevant code:

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple

@dataclass
class UserArgs:
    """User arguments for tweaking the environment"""

    initial_position: Optional[Tuple[float, ...]] = None  # 3-tuple (x, y, theta)
    initial_state: Optional[
        Tuple[float, ...]
    ] = None  # 6-tuple (x, y, x_dot, y_dot, theta, theta_dot)

    initial_barge_position: Optional[Tuple[float, ...]] = None  # 2-tuple (x, theta)

    # render crosses at the rocket center of mass and landing position
    render_landing_position: bool = True
    render_lander_center_position: bool = True

    # disturbances, which should generally be left disabled
    enable_wind: bool = False
    enable_moving_barge: bool = False

    random_initial_position: bool = False

<font color='red'>IMPORTANT</font>: The values $x$ and $y$ are provided as a fraction of the screen width and height, for simplicity. All other values $( \theta, \dot{\theta}, \dot{x}, \dot{y})$ use their normal units of rad, rad/s, and m/s, respectively.

For example, to initialize the rocket a quarter of the way across the screen, halfway up, and at an angle of 0.1 rad, the following arguments could be provided:

In [ ]:
import gymnasium as gym

import coco_rocket_lander
from coco_rocket_lander.env import UserArgs
args: UserArgs = {
    "initial_position": (0.25, 0.5, 0.1)
}

env = gym.make("coco_rocket_lander/RocketLander-v0", render_mode="rgb_array", args=args)

### Other Useful Methods and Parameters

The following additional methods and parameters may be of some use during your implementation.

In [ ]:
gravity = env.unwrapped.cfg.gravity  # fixed at -9.81
mass, inertia = env.unwrapped.get_mass_properties()
l1, l2 = env.unwrapped.get_dimensional_properties()

# normalized main engine thrust to compensate for gravity
gravity_comp_fraction = -gravity * mass / env.unwrapped.cfg.main_engine_thrust

print(f"Gravity: {gravity}")
print(f"Mass, Inertia: {mass, inertia}")
print(f"l1, l2: {l1, l2}")
print(f"Gravity compensation fraction: {gravity_comp_fraction}")